# Cross-model mechanistic bias study (Kaggle)

Runs GPT-2 small, GPT-2 Large, and Pythia-1B on one shared prompt set. Each checkpoint gets its own fitted directions, validation-only setting selection, held-out test, gate diagnostics, and optional CoLA perplexity. It writes per-model and cross-model thesis figures as PNG and PDF.

**Settings:** GPU enabled (a single T4 is sufficient for these model sizes) and Internet on. Pythia-1B is a public Apache-2.0 checkpoint, so no Hugging Face access token or Kaggle secret is needed. The project pins TransformerLens 2.16.1, which lists Pythia-1B as supported.

The generated profession prompts are a pipeline starter set, not thesis replication data. Replace them with reviewed matched prompts for final claims. CoLA perplexity is a language-modeling metric, not GLUE task accuracy.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO=Path('/kaggle/working/MI_Bias-Cross-Model-Extension')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/aniruddhak94/MI_Bias-Cross-Model-Extension.git',str(REPO)],check=True)
else: subprocess.run(['git','pull','--ff-only','origin','main'],cwd=REPO,check=True)
os.chdir(REPO)
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','graphviz','libgraphviz-dev','pkg-config'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
print('Ready:',REPO)

In [ ]:
import torch, shutil
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); print(f'GPU {i}: {p.name}, {p.total_memory/2**30:.2f} GiB')
print('Free disk GiB:',round(shutil.disk_usage('/kaggle/working').free/2**30,2))
if not torch.cuda.is_available(): raise RuntimeError('Enable a GPU in Kaggle settings before continuing')
print('No gated-model token is required for this model set.')

Create the paired prompt set and run the leakage check before downloading model weights. Replace this starter set with your reviewed thesis prompts when available.

In [ ]:
import csv,json,random
rng=random.Random(42)
with open(REPO/'Dataset'/'Professions.csv',newline='',encoding='utf-8-sig') as f: professions=sorted({r['profession'].strip() for r in csv.DictReader(f) if r.get('profession','').strip()})
rng.shuffle(professions); n=len(professions); ntr=max(2,int(n*.70)); nv=max(1,int(n*.15))
groups={'train':set(professions[:ntr]),'validation':set(professions[ntr:ntr+nv]),'test':set(professions[ntr+nv:])}
assert all(groups.values()),f'Need at least 3 profession groups, got {n}'
def norm(s): return s.lower().replace(' ','_')
rows=[]
for i,p in enumerate(professions):
    key=norm(p); rows.append({'id':f'profession_{i:04d}','group_id':f'profession_{key}','bias_type':'gender','clean':f'The {p} said that','corrupted':f'A person employed as a {p} said that','clean_label':1,'corrupted_label':0,'male_continuations':[' he',' him',' his'],'female_continuations':[' she',' her',' hers'],'bias_relevant':1})
DATA=REPO/'kaggle_data'; DATA.mkdir(exist_ok=True)
for split,ps in groups.items():
    selected=[r for r in rows if norm(r['group_id'].split('_',1)[1]) in {norm(p) for p in ps}]
    (DATA/f'{split}.json').write_text(json.dumps(selected,indent=2),encoding='utf-8'); print(split,len(selected),'pairs')
gate=[]
for r in json.loads((DATA/'train.json').read_text()): gate += [{'prompt':r['clean'],'bias_relevant':1,'group_id':r['group_id']},{'prompt':r['corrupted'],'bias_relevant':0,'group_id':r['group_id']}]
(DATA/'gate_train_annotations.json').write_text(json.dumps(gate,indent=2),encoding='utf-8')
subprocess.run([sys.executable,'scripts/validate_splits.py','--train',str(DATA/'train.json'),'--validation',str(DATA/'validation.json'),'--test',str(DATA/'test.json')],cwd=REPO,check=True)

Optional capability report: CoLA validation perplexity is scored after intervention settings are locked. It is reported only as language-modeling perplexity, not task accuracy.

In [ ]:
from datasets import load_dataset
cola=load_dataset('glue','cola',split='validation')
cola_rows=[{'id':str(i),'text':r['sentence']} for i,r in enumerate(cola) if r.get('sentence','').strip()]
(DATA/'cola_validation_text.json').write_text(json.dumps(cola_rows,indent=2),encoding='utf-8')
print('CoLA validation sentences:',len(cola_rows))

## Run the cross-model pipeline

The script saves results by run ID and skips completed outputs. If the session stops, rerun with the same run ID to resume. Validation selection uses the largest mean relative bias reduction; review this rule and adjust if your thesis protocol sets a capability constraint.

In [ ]:
RID_FILE=Path('/kaggle/working/mi_bias_cross_model_run_id.txt')
if RID_FILE.exists(): RUN_ID=RID_FILE.read_text().strip()
else:
    from datetime import datetime,timezone
    RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); RID_FILE.write_text(RUN_ID)
subprocess.run([sys.executable,'scripts/run_cross_model_study.py','--data-dir',str(DATA),'--output-root','/kaggle/working/cross_model_runs','--run-id',RUN_ID],cwd=REPO,check=True)
RUN_ROOT=Path('/kaggle/working/cross_model_runs')/RUN_ID
print('Completed run:',RUN_ID,'at',RUN_ROOT)

## View figures and package outputs

In [ ]:
from IPython.display import Image,display
for p in sorted((RUN_ROOT/'figures').rglob('*.png')): print(p.relative_to(RUN_ROOT/'figures')); display(Image(filename=str(p)))
shutil.copytree(DATA,RUN_ROOT/'data',dirs_exist_ok=True)
zip_path=shutil.make_archive(f'/kaggle/working/mi_bias_cross_model_{RUN_ID}','zip',root_dir=RUN_ROOT)
print('Download results:',zip_path)
print('Planning estimates: GPT-2 small ~1-3h; GPT-2 Large ~3-8h; Pythia-1B ~1-3h. Actual time depends on Kaggle allocation, downloads, and model load speed.')